# vecdb Demo
### Open source hybrid vector database — end-to-end walkthrough

This notebook demonstrates vecdb's full feature set:
- **Dense search** — semantic similarity via HNSW index
- **Sparse search** — keyword relevance via BM25
- **Hybrid search** — fused dense + sparse retrieval
- **SQL queries** — VECTOR_SIM predicate with filters
- **Multi-collection** — isolated namespaces per use case

No GPU or embedding model required — we use hand-crafted 8-dimensional
vectors that make the math easy to follow.

**Prerequisites:** vecdb server running on `http://localhost:6333`  
See `notebooks/README.md` for startup instructions.

## 0. Setup

Import the vecdb Python SDK and verify the server is reachable.

In [ ]:
import sys
import json
import math
import requests

from vecdb import (
    VecDbClient,
    VectorRecord,
    NotFoundError,
    VecDbError,
    CollectionInfo,
)

BASE_URL = "http://localhost:6333"

# Connect to local server
client = VecDbClient(base_url=BASE_URL)

# Verify connectivity
try:
    health = client.health()
    print(f"✓ Connected to vecdb v{health.get('version', '0.1.0')}")
    print(f"  Status      : {health.get('status', 'unknown')}")
    print(f"  Collections : {health.get('collections', 0)}")
    print(f"  Total vectors: {health.get('vector_count', 0)}")
except VecDbError as e:
    print(f"✗ Cannot reach server: {e}")
    print("  Start the server first — see notebooks/README.md")
    sys.exit(1)

## 1. Create a Collection

A collection is an isolated namespace that holds vectors of a fixed dimension.
We use `dimension=8` so the vectors are easy to reason about.
The `cosine` metric measures angular similarity — direction matters, not magnitude.

In [ ]:
COLLECTION = "demo"
DIMENSION = 8

# Clean up from previous runs
try:
    client.delete_collection(COLLECTION)
    print(f"Deleted existing '{COLLECTION}' collection")
except NotFoundError:
    pass

# Create fresh collection
collection = client.create_collection(
    name=COLLECTION,
    dimension=DIMENSION,
    metric="cosine",
)
print(f"\u2713 Created collection '{collection.name}'")
print(f"  Dimension : {collection.dimension}")
print(f"  Metric    : {collection.metric}")
print(f"  Index     : {collection.index_type}")

## 2. Define Documents

We define 20 documents about technology topics. Each document has:
- `id` — unique identifier
- `vector` — 8-dimensional float vector (hand-crafted to cluster by topic)
- `text` — raw text for BM25 keyword search
- `payload` — structured metadata for filtering

The vectors are designed so that ML/AI topics cluster together,
databases cluster together, and systems topics cluster together.

In [ ]:
documents = [
    # ── Machine Learning cluster (high values in dims 0-1) ──────────────────
    {"id": "ml-001",
     "vector": [0.9, 0.8, 0.1, 0.1, 0.0, 0.0, 0.1, 0.0],
     "text": "machine learning neural networks deep learning",
     "payload": {"topic": "ml", "year": 2023, "region": "US"}},
    {"id": "ml-002",
     "vector": [0.8, 0.9, 0.2, 0.0, 0.1, 0.0, 0.0, 0.1],
     "text": "gradient descent backpropagation training neural networks",
     "payload": {"topic": "ml", "year": 2022, "region": "EU"}},
    {"id": "ml-003",
     "vector": [0.7, 0.8, 0.3, 0.1, 0.0, 0.1, 0.0, 0.0],
     "text": "transformer attention mechanism large language models",
     "payload": {"topic": "ml", "year": 2023, "region": "US"}},
    {"id": "ml-004",
     "vector": [0.8, 0.7, 0.1, 0.2, 0.1, 0.0, 0.0, 0.0],
     "text": "reinforcement learning reward policy optimization",
     "payload": {"topic": "ml", "year": 2022, "region": "APAC"}},
    {"id": "ml-005",
     "vector": [0.9, 0.6, 0.2, 0.1, 0.0, 0.0, 0.1, 0.1],
     "text": "computer vision image classification convolutional networks",
     "payload": {"topic": "ml", "year": 2021, "region": "EU"}},

    # ── Database cluster (high values in dims 2-3) ───────────────────────────
    {"id": "db-001",
     "vector": [0.1, 0.1, 0.9, 0.8, 0.0, 0.1, 0.0, 0.0],
     "text": "relational database SQL query optimization indexes",
     "payload": {"topic": "database", "year": 2023, "region": "US"}},
    {"id": "db-002",
     "vector": [0.0, 0.1, 0.8, 0.9, 0.1, 0.0, 0.0, 0.1],
     "text": "vector database similarity search embeddings retrieval",
     "payload": {"topic": "database", "year": 2023, "region": "EU"}},
    {"id": "db-003",
     "vector": [0.1, 0.0, 0.7, 0.8, 0.2, 0.1, 0.0, 0.0],
     "text": "NoSQL document store key value distributed database",
     "payload": {"topic": "database", "year": 2022, "region": "US"}},
    {"id": "db-004",
     "vector": [0.2, 0.1, 0.8, 0.7, 0.0, 0.0, 0.1, 0.0],
     "text": "database indexing B-tree hash index performance",
     "payload": {"topic": "database", "year": 2021, "region": "APAC"}},
    {"id": "db-005",
     "vector": [0.0, 0.2, 0.9, 0.7, 0.1, 0.0, 0.0, 0.0],
     "text": "graph database nodes edges relationships traversal",
     "payload": {"topic": "database", "year": 2022, "region": "EU"}},

    # ── Systems cluster (high values in dims 4-5) ────────────────────────────
    {"id": "sys-001",
     "vector": [0.0, 0.0, 0.1, 0.1, 0.9, 0.8, 0.1, 0.0],
     "text": "operating systems kernel memory management processes",
     "payload": {"topic": "systems", "year": 2021, "region": "US"}},
    {"id": "sys-002",
     "vector": [0.1, 0.0, 0.0, 0.1, 0.8, 0.9, 0.0, 0.1],
     "text": "distributed systems consensus fault tolerance replication",
     "payload": {"topic": "systems", "year": 2023, "region": "EU"}},
    {"id": "sys-003",
     "vector": [0.0, 0.1, 0.1, 0.0, 0.7, 0.8, 0.2, 0.0],
     "text": "network protocols TCP IP routing packet switching",
     "payload": {"topic": "systems", "year": 2022, "region": "US"}},
    {"id": "sys-004",
     "vector": [0.1, 0.0, 0.0, 0.2, 0.8, 0.7, 0.0, 0.1],
     "text": "compiler design lexer parser code generation optimization",
     "payload": {"topic": "systems", "year": 2021, "region": "APAC"}},
    {"id": "sys-005",
     "vector": [0.0, 0.1, 0.2, 0.0, 0.9, 0.7, 0.0, 0.0],
     "text": "concurrency threading locks mutex parallel programming",
     "payload": {"topic": "systems", "year": 2022, "region": "EU"}},

    # ── Cross-topic documents ────────────────────────────────────────────────
    {"id": "cross-001",
     "vector": [0.5, 0.5, 0.5, 0.4, 0.0, 0.0, 0.0, 0.0],
     "text": "machine learning database vector embeddings semantic search",
     "payload": {"topic": "ml", "year": 2023, "region": "US"}},
    {"id": "cross-002",
     "vector": [0.4, 0.3, 0.3, 0.4, 0.5, 0.4, 0.0, 0.0],
     "text": "distributed machine learning training large scale systems",
     "payload": {"topic": "systems", "year": 2023, "region": "EU"}},
    {"id": "cross-003",
     "vector": [0.0, 0.0, 0.4, 0.5, 0.4, 0.5, 0.0, 0.0],
     "text": "database systems storage engine write ahead log",
     "payload": {"topic": "database", "year": 2022, "region": "US"}},
    {"id": "cross-004",
     "vector": [0.6, 0.4, 0.0, 0.0, 0.4, 0.5, 0.0, 0.0],
     "text": "neural network hardware accelerator systems design",
     "payload": {"topic": "systems", "year": 2023, "region": "APAC"}},
    {"id": "cross-005",
     "vector": [0.3, 0.4, 0.5, 0.3, 0.3, 0.3, 0.0, 0.0],
     "text": "hybrid search dense sparse retrieval information retrieval",
     "payload": {"topic": "database", "year": 2023, "region": "EU"}},
]

print(f"Defined {len(documents)} documents across 3 topic clusters:")
for topic in ["ml", "database", "systems"]:
    count = sum(1 for d in documents if d["payload"]["topic"] == topic)
    label = {"ml": "ML cluster     ", "database": "Database cluster", "systems": "Systems cluster "}[topic]
    print(f"  {label}: {count} docs")

## 3. Upsert Documents

Upsert inserts new documents or updates existing ones by id.
We L2-normalize each vector before inserting — required for cosine similarity
to work correctly (cosine compares directions, not magnitudes).
All 20 documents are sent in a single batch.

In [ ]:
def normalize(v):
    """L2-normalize a vector so cosine similarity works correctly."""
    mag = math.sqrt(sum(x * x for x in v))
    return [x / mag for x in v] if mag > 0 else v


records = [
    VectorRecord(
        id=doc["id"],
        vector=normalize(doc["vector"]),
        text=doc["text"],
        payload=doc["payload"],
    )
    for doc in documents
]

result = client.upsert(COLLECTION, records)
print(f"\u2713 Upsert complete")
print(f"  Inserted : {result.inserted}")
print(f"  Updated  : {result.updated}")
print(f"  Errors   : {result.errors}")
print(f"  Time     : {result.time_ms:.1f}ms")

## 4. Dense Search (Semantic Similarity)

Dense search uses the HNSW index to find vectors with the highest
cosine similarity to the query. It captures semantic meaning —
documents about similar topics cluster together regardless of exact wording.

**Query:** pointing toward the ML cluster (`[0.85, 0.80, ...]`)  
**Expected:** ML cluster documents should rank highest.

In [ ]:
# Query vector pointing toward the ML cluster (high dims 0-1)
query_ml = normalize([0.85, 0.80, 0.15, 0.10, 0.05, 0.05, 0.05, 0.05])

results = client.search_dense(COLLECTION, query_ml, k=5)

print(f"Dense search results (k=5) \u2014 {results.time_ms:.1f}ms")
print(f"{'ID':<12} {'Score':>8} {'Topic':<12} Text snippet")
print("\u2500" * 72)
for r in results.results:
    topic = r.payload.get("topic", "?") if r.payload else "?"
    snippet = (r.text or "")[:40]
    print(f"{r.id:<12} {r.score:>8.4f} {topic:<12} {snippet}")

## 5. Sparse Search (BM25 Keyword Relevance)

Sparse search uses the BM25 inverted index to find documents containing
the query terms. It rewards exact keyword matches and penalizes common words.

**Query:** `"database indexing performance"`  
**Expected:** Database cluster documents with those exact terms rank highest.

In [ ]:
results = client.search_sparse(COLLECTION, "database indexing performance", k=5)

print(f"Sparse search results (k=5) \u2014 {results.time_ms:.1f}ms")
print(f"{'ID':<12} {'Score':>8} {'Topic':<12} Text snippet")
print("\u2500" * 72)
for r in results.results:
    topic = r.payload.get("topic", "?") if r.payload else "?"
    snippet = (r.text or "")[:40]
    print(f"{r.id:<12} {r.score:>8.4f} {topic:<12} {snippet}")

## 6. Hybrid Search (Dense + Sparse Fusion)

Hybrid search combines both signals. The `alpha` parameter controls the blend:
- `alpha=1.0` \u2192 pure dense (semantic only)
- `alpha=0.0` \u2192 pure sparse (keyword only)
- `alpha=0.7` \u2192 70% dense, 30% sparse (default, usually best)

We search for `"vector database similarity"` with both a semantic query vector
and keyword terms. Documents that score well on **both** signals rank highest.

In [ ]:
# Query toward the database cluster with some ML overlap (high dims 2-3)
query_db = normalize([0.2, 0.2, 0.8, 0.8, 0.1, 0.1, 0.0, 0.0])

results = client.search_hybrid(
    COLLECTION,
    vector=query_db,
    query="vector database similarity search embeddings",
    k=5,
    alpha=0.7,
)

print(f"Hybrid search results (alpha=0.7, k=5) \u2014 {results.time_ms:.1f}ms")
print(f"{'ID':<12} {'Score':>8} {'Dense':>8} {'Sparse':>8} {'Topic'}")
print("\u2500" * 62)
for r in results.results:
    topic = r.payload.get("topic", "?") if r.payload else "?"
    dense = f"{r.dense_score:.4f}" if r.dense_score is not None else "     -"
    sparse = f"{r.sparse_score:.4f}" if r.sparse_score is not None else "     -"
    print(f"{r.id:<12} {r.score:>8.4f} {dense:>8} {sparse:>8} {topic}")

In [ ]:
# Show how alpha changes the ranking
print("Effect of alpha on hybrid search ranking:\n")
alpha_labels = {
    1.0: "pure dense   ",
    0.7: "mostly dense ",
    0.3: "mostly sparse",
    0.0: "pure sparse  ",
}
for alpha in [1.0, 0.7, 0.3, 0.0]:
    res = client.search_hybrid(
        COLLECTION,
        vector=query_db,
        query="vector database similarity search embeddings",
        k=3,
        alpha=alpha,
    )
    top_ids = [r.id for r in res.results]
    print(f"  alpha={alpha:.1f} ({alpha_labels[alpha]}): {top_ids}")

## 7. SQL Query Language

vecdb supports a SQL-like query language with a `VECTOR_SIM` predicate.
This lets you combine semantic search with structured metadata filters
in a single query — no post-processing needed.

Supported features:
- `VECTOR_SIM(vec, [...]) > threshold` — similarity threshold filter
- `AND field = 'value'` — equality filter on payload fields
- `AND field > value` — numeric comparison
- `ORDER BY score DESC` — sort by relevance
- `LIMIT N` — cap result count

In [ ]:
# Example 1: Vector similarity with score threshold
sql1 = (
    "SELECT * FROM demo "
    "WHERE VECTOR_SIM(vec, [0.2, 0.2, 0.8, 0.8, 0.1, 0.1, 0.0, 0.0]) > 0.5 "
    "LIMIT 5"
)
r1 = client.query_sql(sql1)
print(f"SQL 1 — VECTOR_SIM > 0.5: {len(r1.results)} results")
for r in r1.results:
    topic = r.payload.get("topic", "?") if r.payload else "?"
    print(f"  {r.id:<12} score={r.score:.4f}  topic={topic}")

print()

# Example 2: Vector similarity + payload filter
sql2 = (
    "SELECT * FROM demo "
    "WHERE VECTOR_SIM(vec, [0.85, 0.80, 0.15, 0.10, 0.05, 0.05, 0.05, 0.05]) > 0.3 "
    "AND payload->>'region' = 'US' "
    "LIMIT 5"
)
r2 = client.query_sql(sql2)
print(f"SQL 2 — VECTOR_SIM + region='US': {len(r2.results)} results")
for r in r2.results:
    region = r.payload.get("region", "?") if r.payload else "?"
    print(f"  {r.id:<12} score={r.score:.4f}  region={region}")

print()

# Example 3: Vector similarity + numeric payload filter
sql3 = (
    "SELECT * FROM demo "
    "WHERE VECTOR_SIM(vec, [0.2, 0.2, 0.8, 0.8, 0.1, 0.1, 0.0, 0.0]) > 0.0 "
    "AND payload->>'year' > 2022 "
    "LIMIT 10"
)
r3 = client.query_sql(sql3)
print(f"SQL 3 — VECTOR_SIM + year > 2022: {len(r3.results)} results")
for r in r3.results:
    year = r.payload.get("year", "?") if r.payload else "?"
    topic = r.payload.get("topic", "?") if r.payload else "?"
    print(f"  {r.id:<12} year={year}  topic={topic}")

## 8. Collection Inspection

Check the collection stats and list all collections on the server.
The `get_collection` call returns live counts — no staleness.

In [ ]:
info = client.get_collection(COLLECTION)
print(f"Collection: {info.name}")
print(f"  Vectors  : {info.vector_count}")
print(f"  Dimension: {info.dimension}")
print(f"  Metric   : {info.metric}")
print(f"  Index    : {info.index_type}")
print(f"  Created  : {info.created_at}")

print()

# List all collections via the REST API directly.
# The server returns {"collections": [...], "count": N}.
resp = requests.get(f"{BASE_URL}/collections")
data = resp.json()
raw = data.get("collections", []) if isinstance(data, dict) else data
all_collections = [CollectionInfo.from_dict(c) for c in raw]

print(f"All collections on this server: {len(all_collections)}")
for c in all_collections:
    print(f"  {c.name:<20} dim={c.dimension}  vectors={c.vector_count}")

## 9. Get and Delete Vectors

Fetch a specific vector by ID (point lookup), then delete it and confirm
it is gone. `NotFoundError` is raised on any get/search of a deleted vector.

In [ ]:
# Fetch a specific vector by ID
record = client.get_vector(COLLECTION, "ml-001")
print(f"Fetched record: {record.id}")
print(f"  Vector (first 4 dims): {[round(x, 4) for x in record.vector[:4]]}")
print(f"  Text   : {record.text}")
print(f"  Payload: {record.payload}")

print()

# Delete it
client.delete_vectors(COLLECTION, ["ml-001"])
print("Deleted ml-001")

# Confirm deletion
try:
    client.get_vector(COLLECTION, "ml-001")
    print("ERROR: vector still exists!")
except NotFoundError:
    print("\u2713 Confirmed: ml-001 is gone")

# Dense search no longer returns ml-001
results = client.search_dense(COLLECTION, query_ml, k=5)
ids_after_delete = [r.id for r in results.results]
assert "ml-001" not in ids_after_delete, "ml-001 should not appear in search after delete"
print(f"\u2713 ml-001 absent from dense search results: {ids_after_delete}")

## 10. Cleanup

Delete the demo collection to leave the server in a clean state.
Safe to skip if you want to keep exploring the data.

In [ ]:
client.delete_collection(COLLECTION)
print(f"\u2713 Deleted collection '{COLLECTION}'")

# Verify it's gone
try:
    client.get_collection(COLLECTION)
    print("ERROR: collection still exists!")
except NotFoundError:
    print("\u2713 Server is clean")

## Summary

You have seen the full vecdb feature set end-to-end:

| Feature | What we did |
|---|---|
| Collection management | Create, inspect, delete |
| Vector upsert | Batch insert 20 documents |
| Dense search | HNSW cosine similarity, top-5 results |
| Sparse search | BM25 keyword scoring |
| Hybrid search | Weighted fusion with alpha tuning |
| SQL queries | VECTOR_SIM with metadata filters |
| Get / Delete | Point lookups and soft deletes |

### Next steps

- **Real embeddings**: replace hand-crafted vectors with a sentence transformer  
  (`sentence-transformers/all-MiniLM-L6-v2` → dim=384)
- **Large scale**: use `vecdb-bench` to benchmark with 10k\u20131M vectors
- **Production**: run with Docker, set `VECDB__API_KEY` for authentication
- **IVF index**: pass `index_type="ivf"` to `create_collection` for large collections
- **SDKs**: see `sdks/python/` and `sdks/typescript/` for full API coverage

### Useful links

- [API Reference](../docs/api.md)
- [Architecture](../docs/architecture.md)
- [Configuration](../docs/configuration.md)
- [Benchmark scripts](../benchmarks/README.md)